In [15]:
from flask import Flask, request, jsonify
import mlflow.pyfunc
import pandas as pd
import os

app = Flask(__name__)

# Get the latest logged model from MLflow
def get_latest_model():

    # Get the MLflow tracking URI from the environment variable
    mlflow_tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")

    if mlflow_tracking_uri is None:
        # Handle the case where the environment variable is not set
        print("Error: MLFLOW_TRACKING_URI environment variable not set.")
        return None  # Or raise an exception

    # Set the tracking URI for MLflow
    mlflow.set_tracking_uri(mlflow_tracking_uri)

    runs = mlflow.search_runs(filter_string="metrics.accuracy > 0.5", order_by=["metrics.accuracy DESC"], max_results=1) # Set your threshold

    if runs.empty:
        return None

    run_id = runs.iloc[0]["run_id"]
    model_uri = f"runs:/{run_id}/model"
    return mlflow.pyfunc.load_model(model_uri)

model = get_latest_model()

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        # Convert the incoming JSON data to a Pandas DataFrame
        data_df = pd.DataFrame(data)

        # Make predictions using the loaded model
        predictions = model.predict(data_df)

        # Convert predictions to a list before returning as JSON
        return jsonify(predictions.tolist())

    except Exception as e:
        return jsonify({"error": str(e)})

if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=5001)

 * Serving Flask app '__main__'
 * Debug mode: on



 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.17.0.2:5001
Press CTRL+C to quit
 * Restarting with stat
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.11/site-packages/traitlets/config/application.py", line 1079, in launch_instance
    app.initialize(argv)
  File "/usr/local/lib/python3.11/site-packages/traitlets/config/application.py", line 117, in inner
    return method(app, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 692, in initialize
    self.init_sockets()
  File "/usr/local/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 331, in init_sockets
    self.shell_port = self._bind_socket(self.shell_socket, self.shell_port)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

SystemExit: 1

/usr/local/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


SyntaxError: invalid syntax (2009670201.py, line 1)